In [1]:
import os
os.makedirs("results", exist_ok=True)

import numpy as np, pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

expression = pd.read_csv("../assignment3/shared_expression_log2.csv", index_col=0)
sample_meta = pd.read_csv("../assignment3/shared_sample_metadata.csv", index_col=0)
hier = pd.read_csv("../assignment3/results/hier_cluster_assignments.csv", index_col=0)
assert (expression.index == sample_meta.index).all() and (hier.index == sample_meta.index).all()


In [2]:
# same variance ranking as Assignment 3
ranked = expression.var(axis=0).sort_values(ascending=False).index

# 4 folds, stratified on condition x age x genotype
strat = sample_meta["condition"] + "*" + sample_meta["age"] + "*" + sample_meta["genotype"]
folds = pd.Series(-1, index=sample_meta.index, name="fold")
for i, (_, test_idx) in enumerate(StratifiedKFold(4, shuffle=True, random_state=42).split(expression, strat)):
    folds.iloc[test_idx] = i

print(folds.value_counts())
folds.to_csv("results/cv_folds.csv")
cv = [(np.where(folds != i)[0], np.where(folds == i)[0]) for i in range(4)]


fold
1    8
3    8
0    8
2    8
Name: count, dtype: int64


In [3]:
y_cond = (sample_meta["condition"] == "flight").astype(int)
y_clust = hier["hier_k3"]
targets = {"condition": y_cond, "cluster_k3": y_clust}


In [4]:
def run_nb(n_genes, y):
    X = expression[ranked[:n_genes]]
    proba = cross_val_predict(GaussianNB(), X, y, cv=cv, method="predict_proba")
    classes = np.sort(y.unique())
    pred = classes[proba.argmax(axis=1)]
    if len(classes) == 2:
        auc = roc_auc_score(y, proba[:, 1])
    else:
        auc = roc_auc_score(y, proba, multi_class="ovr")
    return pred, auc

rows, preds = [], {}
for n in [10, 100, 1000, 5000, 10000]:
    for name, y in targets.items():
        pred, auc = run_nb(n, y)
        rows.append({"Method": "Naive Bayes", "Genes": n, "Target": name,
                     "AUC": auc, "Accuracy": (pred == y.values).mean()})
        preds[f"nb_{name}_{n}genes"] = pred

auc_table = pd.DataFrame(rows)
auc_table.to_csv("results/nb_auc_by_gene_count.csv", index=False)
pd.DataFrame(preds, index=expression.index).to_csv("results/nb_predictions.csv")
auc_table


,Method,Genes,Target,AUC,Accuracy
0,Naive Bayes,10,condition,0.531250,0.56250
1,Naive Bayes,10,cluster_k3,0.807182,0.62500
2,Naive Bayes,100,condition,0.703125,0.62500
3,Naive Bayes,100,cluster_k3,0.878803,0.84375
4,Naive Bayes,1000,condition,0.630859,0.59375
5,Naive Bayes,1000,cluster_k3,0.971678,0.96875
6,Naive Bayes,5000,condition,0.593750,0.59375
7,Naive Bayes,5000,cluster_k3,0.946215,0.93750
8,Naive Bayes,10000,condition,0.562500,0.56250
9,Naive Bayes,10000,cluster_k3,0.946215,0.93750
